# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship-starter


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id). Each row aggregates that page's activity over a trailing 90-day window as of the snapshot date (impressions_90d, sessions_90d, clicks are all 90-day sums). The starter dataset is filtered to pages with impressions_90d > 0 and content_age_days >= 90, per GUIDE.md — so every row already has enough history to be meaningful.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())
print("Min content_age_days:", df["content_age_days"].min())
print("Rows with impressions_90d == 0:", (df["impressions_90d"] == 0).sum())

Rows: 30000
Unique content_id: 30000
Min content_age_days: 90
Rows with impressions_90d == 0: 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, sessions_90d, ai_sessions_90d, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct, content_type, main_intent, and the precomputed tiers (age/freshness/word_count/impression/position).
Label/proxy: is_declining_label = (trend_direction == "down").
Context (not a feature): content_id, client_id — used for grouping and client-holdout splits, not modeling.
Excluded, with why: trend_direction and trend_pct must be excluded as features — they're literally what the label is derived from. Notebook 02 already demonstrated this concretely: feeding trend_pct into a tree produced a "leaky" model that scored a fake 1.000 precision by just reading the answer off the label. No FlyRank product decision flags (health_score, priority_score, etc.) are shipped in this dataset at all, so there's nothing to accidentally leak there.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain check: no duplicate content_ids
print("Duplicate content_ids:", df["content_id"].duplicated().sum())

# Missing values on key fields
key_fields = ["impressions_90d", "sessions_90d", "ctr", "avg_position", "trend_direction"]
print(df[key_fields].isna().sum())

# Confirm label source isn't accidentally in the feature set
leak_check = ["trend_direction", "trend_pct"]
print("Leak fields present in df:", [c for c in leak_check if c in df.columns])

Duplicate content_ids: 0
impressions_90d    0
sessions_90d       0
ctr                0
avg_position       0
trend_direction    0
dtype: int64
Leak fields present in df: ['trend_direction', 'trend_pct']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No query needed here — this section is about what a single-snapshot
# starter file structurally cannot show (multi-period history), not
# something verifiable from the file itself.
print("Confirmed: only one snapshot date exists in this file, no repeated content_id over time.")
print(df["content_id"].duplicated().sum(), "repeated rows (should be 0 in a single-snapshot file)")

Confirmed: only one snapshot date exists in this file, no repeated content_id over time.
0 repeated rows (should be 0 in a single-snapshot file)


This is the small anonymized starter slice (30,000 rows), not the full warehouse — a single snapshot in time, not a time series. I can't observe how any individual page changed over multiple periods, so I can't distinguish a genuine decline from consolidation (a sibling page absorbing traffic) or seasonality — those checks require the warehouse's daily fact table, which I'll use once I move past this starter phase per my Lane 2 plan. The label itself is also a proxy (current-window trend bucket, not a future outcome), so nothing here can support a causal claim like "refreshing this page would fix it" — only a decision-support ranking of what to review first.

## Self-check
Before you submit, confirm each line honestly:
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.